# 02 · Results

Every trained run on the official test split, then generalisation to unseen recordings and the explanations. Write-ups: `results/baselines.md`, `results/improved_model.md`, `results/cross_session.md`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
RES = ROOT / "results"


## All runs, official test split

NAR = share of alarms that were really background; FNR = share of events classified as background (the paper's definitions). Paper: SVM 82.6%, CNN 94.0%.

In [2]:
runs = pd.read_csv(RES / "summary_table.csv")
runs[["run", "accuracy", "macro_f1", "nar", "fnr", "dig_miss_rate", "params", "latency_ms"]]

,run,accuracy,macro_f1,nar,fnr,dig_miss_rate,params,latency_ms
0,net1d_dig2_chnorm,0.9919,0.9916,0.0004,0.0008,0.0060,181254.0,1.2733
1,net1d,0.9906,0.9902,0.0008,0.0008,0.0120,181254.0,2.6118
2,net1d_dig2,0.9903,0.9899,0.0012,0.0008,0.0080,181254.0,3.9190
3,net1d_noaug,0.9896,0.9892,0.0012,0.0004,0.0139,181254.0,3.8907
4,cnn_bilstm,0.9880,0.9875,0.0004,0.0016,0.0159,151943.0,1.8045
5,svm_tuned,0.9880,0.9875,0.0004,0.0012,0.0100,759717.0,22.6732
6,spec_cnn,0.9854,0.9850,0.0028,0.0004,0.0219,104734.0,4.2011
7,net1d_dig2_crop,0.9799,0.9792,0.0016,0.0020,0.0139,181254.0,1.2899
8,paper_cnn_long,0.9322,0.9302,0.0052,0.0092,0.0637,4721.0,0.2851
9,svm_paper,0.8803,0.8789,0.0004,0.0645,0.2390,NaN,NaN


![trade-off](../../results/summary_tradeoff.png)

![curves](../../results/summary_curves.png)

## Generalisation: unseen sessions (5-fold) and unseen days (3-fold)

The official split shares recording sessions between train and test; holding out whole sessions / days shows what survives in a new deployment.

In [3]:
sess = pd.read_csv(RES / "cross_session.csv").set_index("model")
days = pd.read_csv(RES / "cross_date.csv").set_index("model")
pd.DataFrame({"unseen sessions": sess.accuracy_mean, "± sessions": sess.accuracy_std,
              "unseen days": days.accuracy_mean, "± days": days.accuracy_std,
              "dig miss, unseen days": days.dig_miss_rate_mean})

,unseen sessions,± sessions,unseen days,± days,"dig miss, unseen days"
model,,,,,
svm,0.9853,0.0050,0.7377,0.1032,0.3535
paper_cnn,0.9143,0.0139,0.6474,0.0264,0.4144
net1d_dig2,0.9780,0.0036,0.7279,0.1050,0.2409
net1d_dig2_chnorm,0.9907,0.0029,0.8011,0.0825,0.2312


The deployed model (`net1d_dig2_chnorm`, each fiber point z-scored separately) is the most robust. Most of the day-level digging misses come from the two spring digging days, which fell in the same fold.

In [4]:
cm = np.array(json.loads((RES / "cross_date_net1d_dig2_chnorm.json").read_text())["confusion_matrix"])
classes = ["background", "dig", "knock", "water", "shake", "walk"]
pd.DataFrame(cm, index=[f"true {c}" for c in classes], columns=classes)

,background,dig,knock,water,shake,walk
true background,2783,42,1,110,0,9
true dig,74,1943,81,55,7,352
true knock,37,33,2420,2,1,37
true water,22,24,0,1779,401,27
true shake,4,8,0,827,1762,127
true walk,50,517,105,115,45,1618


## Deployed model on the official split

![confusion](../../results/confusion_net1d_dig2_chnorm.png)

## Explainability

Grad-CAM (when) and Integrated Gradients (where + when) for one correctly classified test sample per class:

![dig](../../results/explain/net1d_dig2_chnorm_dig.png)

![knock](../../results/explain/net1d_dig2_chnorm_knock.png)

![points](../../results/explain/net1d_dig2_chnorm_point_importance.png)

**Faithfulness (deletion test):** replacing the window Grad-CAM marks as important with real background lowers the predicted-class probability more than replacing a random window of the same length.

In [5]:
f = json.loads((RES / "explain/net1d_dig2_chnorm_faithfulness.json").read_text())
pd.DataFrame({k: {"Grad-CAM window": v["prob_drop_gradcam"], "random window": v["prob_drop_random"]} for k, v in f["by_fraction"].items()}).T.rename_axis("share of window replaced")

,Grad-CAM window,random window
share of window replaced,,
0.1,0.1841,0.1437
0.2,0.3389,0.2467
0.3,0.4990,0.3665


## SVM feature importance (SHAP via a random-forest surrogate)

![shap](../../results/explain/shap_by_feature_type.png)

In [6]:
s = json.loads((RES / "explain/shap_summary.json").read_text())
print("surrogate agrees with the SVM on", f"{s['surrogate_fidelity']:.1%}", "of test samples")
pd.Series(s["top_features"]).head(10)

surrogate agrees with the SVM on 98.6% of test samples


p05_diff_kurtosis_factor     0.0135
p06_diff_kurtosis_factor     0.0103
p05_diff_kurtosis            0.0097
p06_diff_kurtosis            0.0089
p05_diff_peak_to_peak        0.0061
p06_raw_kurtosis             0.0057
p05_diff_spectral_entropy    0.0052
p05_diff_std                 0.0050
p05_diff_arv                 0.0048
p06_diff_spectral_entropy    0.0048
dtype: float64